### Demo Masked Attention

#### Imports

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple
                           

#### Configurations

In [28]:
@dataclass
class Config:
    line_divider: str = '-' * 50
    embed_dim: int = 2
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    small_value: float = 1e-9  # To prevent division by zero in softmax
    random_seed: int = 42  # For reproducibility
    
config = Config()                                                                                              

#### Self-Attention Class

In [29]:
torch.manual_seed(config.random_seed)

class MaskedAttention(nn.Module):
    def __init__(self, d_model=config.embed_dim,  
                 row_dim=0, 
                 col_dim=1):
        super(MaskedAttention, self).__init__()
        self.d_model = d_model
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)  
        self.W_v = nn.Linear(d_model, d_model)
        self.row_dim = row_dim
        self.col_dim = col_dim
        

    def forward(
        self, 
        token_encodings, 
        mask: Optional[torch.Tensor] = None
        ) -> Tuple[torch.Tensor, torch.Tensor]:
        
        Q = self.W_q(token_encodings)  
        K = self.W_k(token_encodings)  
        V = self.W_v(token_encodings)  
        scores = torch.matmul(Q, K.transpose(dim0=self.row_dim, dim1= self.col_dim))
        scores_scaled = scores / (self.d_model ** 0.5)
        print("Scores (Before masking):", scores_scaled)
        print(f"{config.line_divider}\n")
        if mask is not None:
            scores_scaled = scores_scaled.masked_fill(mask == 0, value=config.small_value)
            print("Scores (After masking):", scores_scaled)
            print(f"{config.line_divider}\n")
        attn_weights = F.softmax(scores_scaled, dim=self.col_dim)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

#### Calculate Masked-Attention

In [27]:
embed_dim = config.embed_dim
text = "The cat sat on the mat, it was black."
tokens = text.split()
n_tokens = len(tokens)
token_encodings = torch.randn(len(tokens), embed_dim)
self_attention = MaskedAttention(d_model=embed_dim)
mask = mask = torch.tril(torch.ones(n_tokens, n_tokens)).to(config.device)
output, attn_weights = self_attention(token_encodings, mask=mask)
print("Output shape:", output.shape)
print("Attention weights shape:", attn_weights.shape)
print(f"{config.line_divider}\n")
print("Output:", output)
print(f"{config.line_divider}\n")
print("Attention weights:", attn_weights)
print(f"{config.line_divider}\n")


Scores (Before masking): tensor([[-0.1407, -0.6725, -0.6273, -0.5920, -0.2546, -0.8668, -1.0561, -0.5216,
         -0.6551],
        [ 0.1910,  0.0597,  0.0729,  0.3471,  0.1262,  0.1211,  0.0592,  0.1563,
          0.3712],
        [ 0.1590, -0.0079,  0.0081,  0.2582,  0.0903,  0.0295, -0.0435,  0.0928,
          0.2741],
        [-0.3445, -0.7451, -0.7133, -0.9674, -0.3916, -1.0071, -1.1337, -0.6941,
         -1.0568],
        [-0.0032, -0.4207, -0.3842, -0.2304, -0.1100, -0.5212, -0.6770, -0.2740,
         -0.2610],
        [ 0.1137,  0.0435,  0.0507,  0.2110,  0.0772,  0.0819,  0.0480,  0.0982,
          0.2259],
        [ 0.2592,  0.3434,  0.3384,  0.6116,  0.2388,  0.4890,  0.5030,  0.3818,
          0.6633],
        [-0.0107, -0.3020, -0.2766, -0.1793, -0.0839, -0.3762, -0.4843, -0.2019,
         -0.2022],
        [-0.3771, -0.7612, -0.7311, -1.0298, -0.4147, -1.0350, -1.1533, -0.7246,
         -1.1238]], grad_fn=<DivBackward0>)
--------------------------------------------------